# 파생변수 v3 — **노트북 B (CPU): 트리·선형 v3/v2+v3 멤버 생성기**
역할 변경: 더 이상 자체 채택/제출 안 함. **스크리닝(필터) → 통과 (후보×멤버)의 v3·v2+v3 뷰 OOF/test 생성** → 그리드 노트북이 모델별 `{base, v2, v3, v2+v3}`(선형 `{base, ratio, v3, ratio+v3}`)를 골라 최종 결정.

**왜 이 구조:** base→v3 증분은 "v3에 신호 있나"(필터)일 뿐, "v2+v3가 v2를 이기나"(결정)가 아님. 결정은 **운영(v2 포함) 블렌드 위에서** 그리드가 함. 그래서 B는 v3 뿐 아니라 **v2 위에 얹은 v2+v3 뷰도** 만들어야 그리드가 비교 가능.

**누수:** 신규파생·v2게이팅·lin비율 전부 행단위 또는 fold-내부 fit. test는 train fit→predict.
**입력(Add Input):** train/test + 커밋 `oof_predictions.csv`(STEP1·스크리닝 base 정렬 확인용).
**산출:** `oof_v3_trees`·`test_v3_trees`·`oof_v2v3_trees`·`test_v2v3_trees`·`oof_lin_v3`·`test_lin_v3`·`oof_lin_ratio_v3`·`test_lin_ratio_v3` + `v3b_screen.json`.

## 0. 설정 + DRYRUN (full rows, 가벼운 iter·1시드)

In [1]:
import os,glob,re,json,time,warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore")
DRYRUN=False
SEEDS=(42,) if DRYRUN else (42,7,2024)     # 멤버 생성 시드(그리드는 seed42 기준, 멀티시드는 확인용)
TREE_ITERS=200 if DRYRUN else 1500
def find_csv(n):
    h=[p for p in glob.glob("/kaggle/input/**/*.csv",recursive=True) if os.path.basename(p)==n]; assert h,f"{n} 없음"; return sorted(h,key=len)[0]
train=pd.read_csv(find_csv("train.csv")); test=pd.read_csv(find_csv("test.csv"))
TARGET="임신 성공 여부"; ID_COL="ID"; y=train[TARGET].astype(int).values; N=len(train); base_rate=y.mean()
def NUM(df,c): return pd.to_numeric(df[c],errors="coerce") if c in df else pd.Series(np.nan,index=df.index)
def DIV(num,den): den=den.astype(float); return num.astype(float)/den.where(den>0)
def runr(x): return rankdata(x)/len(x)
COL_PROC="특정 시술 유형"; COL_RSN="배아 생성 주요 이유"
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
OCC=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
CMAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
_tp=lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]",str(s)) if t.strip()]
print("train",train.shape,"| base_rate=%.4f | DRYRUN=%s | SEEDS=%s | TREE_ITERS=%d"%(base_rate,DRYRUN,SEEDS,TREE_ITERS))

train (256351, 69) | base_rate=0.2583 | DRYRUN=False | SEEDS=(42, 7, 2024) | TREE_ITERS=1500


## 0b. 정본 — tf_tree(트리) + v2게이팅 + lin비율 + 신규파생 + 멤버별 prep

In [2]:
def fit_tree(tr):
    st={}; ig={TARGET,ID_COL}
    st["dead"]=[c for c in tr.columns if c not in ig and tr[c].nunique(dropna=True)<=1]
    st["sparse"]=[c for c in tr.columns if c not in ig and c not in st["dead"] and tr[c].isna().mean()>0.98]
    st["lc"]={c:pd.Index(tr[c].astype("category").cat.categories) for c in NOMINAL_COLS if c in tr}
    st["pv"]=sorted({t for L in tr[COL_PROC].apply(_tp) for t in L}); return st
def tf_tree(df,st):
    df=df.copy()
    if TARGET in df: df=df.drop(columns=[TARGET])
    df["is_DI"]=(df["시술 유형"]=="DI").astype(int)
    df=df.drop(columns=[c for c in st["dead"] if c in df.columns])
    for c in st["sparse"]:
        if c in df: df[f"{c}_있음"]=df[c].notna().astype(int); df=df.drop(columns=[c])
    for c in OCC:
        if c in df: df[c]=df[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].astype(object).map(m)
    cats=[]
    for c,cc in st["lc"].items():
        if c in df: df[c]=pd.Categorical(df[c],categories=cc).codes.astype("int32"); cats.append(c)
    ts=df[COL_PROC].apply(_tp)
    for v in st["pv"]: df[f"proc_{v}"]=ts.apply(lambda L,v=v:int(v in L))
    df=df.drop(columns=[c for c in [COL_PROC,COL_RSN,ID_COL] if c in df.columns])
    obj=[c for c in df.columns if df[c].dtype==object]
    if obj: df=df.drop(columns=obj)
    for c in cats: df[c]=df[c].fillna(-1).astype("int32")
    return df,[c for c in cats if c in df.columns]
st=fit_tree(train); Xb,CATF=tf_tree(train,st); Xb_te,_=tf_tree(test,st); Xb_te=Xb_te.reindex(columns=Xb.columns)
# ── v2 게이팅 정본 (3tree-fe-062202와 동일)
def masks(df):
    return {"신선":NUM(df,"신선 배아 사용 여부")==1,"동결":NUM(df,"동결 배아 사용 여부")==1,
            "기증배아":NUM(df,"기증 배아 사용 여부")==1,"DI":(df["시술 유형"].astype(str)=="DI"),
            "ICSI":NUM(df,"미세주입된 난자 수")>0,
            "본인난자":df["난자 출처"].astype(str)=="본인 제공","기증난자":df["난자 출처"].astype(str)=="기증 제공"}
def v1_ratios(df):
    return {"P1":DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")),
            "P2":DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수")),
            "P6":DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")),
            "L3":NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일"),
            "L6":DIV(NUM(df,"총 생성 배아 수"),NUM(df,"미세주입된 난자 수"))}
def build_v2_gated(df):
    Mk=masks(df); r=v1_ratios(df); F={}
    F["g신선_수정률"]=r["P1"].where(Mk["신선"]); F["gICSI_수정효율"]=r["P2"].where(Mk["ICSI"])
    F["g본인_난자수율"]=r["P6"].where(Mk["본인난자"]); F["g기증_난자수율"]=r["P6"].where(Mk["기증난자"])
    F["g신선_배양일수"]=r["L3"].where(Mk["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(Mk["동결"])
    F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    F["FZ3_해동난자수율"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"해동 난자 수"))
    F["PG1_PGT강도"]=NUM(df,"착상 전 유전 검사 사용 여부").fillna(0)+NUM(df,"착상 전 유전 진단 사용 여부").fillna(0)
    F["PG2_PGT분류"]=NUM(df,"PGD 시술 여부").fillna(0)+NUM(df,"PGS 시술 여부").fillna(0)
    F["ST1_자극"]=NUM(df,"배란 자극 여부").fillna(0)
    return pd.DataFrame(F,index=df.index)
V2tr=build_v2_gated(train); V2te=build_v2_gated(test)
# ── lin 비율 정본 (lr-fe와 동일)
def build_lin_ratios(df):
    Mk=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["P1_수정률"]=P1; F["P2_ICSI수정"]=P2; F["P6_난자수율"]=P6; F["L3_배양일수"]=L3
    F["P3_활용률"]=DIV(NUM(df,"이식된 배아 수")+NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수")); F["P5_저장률"]=DIV(NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수"))
    F["L6_배아perICSI난자"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"미세주입된 난자 수")); F["N6_난자감모"]=DIV(NUM(df,"수집된 신선 난자 수")-NUM(df,"혼합된 난자 수"),NUM(df,"수집된 신선 난자 수"))
    F["g신선_수정률"]=P1.where(Mk["신선"]); F["gICSI_수정효율"]=P2.where(Mk["ICSI"]); F["g본인_수율"]=P6.where(Mk["본인난자"]); F["g기증_수율"]=P6.where(Mk["기증난자"]); F["g신선_배양일수"]=L3.where(Mk["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(Mk["동결"]); F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    return pd.DataFrame(F,index=df.index)
RATtr=build_lin_ratios(train); RATte=build_lin_ratios(test)
# ── 신규 v3 파생
def build_new_derived(df):
    F={}
    tx =NUM(df,"이식된 배아 수").fillna(0); sto=NUM(df,"저장된 배아 수").fillna(0); emb=NUM(df,"총 생성 배아 수").fillna(0)
    ses=(df["단일 배아 이식 여부"]==1).values
    F["EL_set_type"]=np.where(~ses,0,np.where(sto.values>0,2,1)).astype("int8")
    F["FA_no_transfer"]=(tx==0).astype("int8").values
    is_current=df[COL_RSN].astype(str).str.contains("현재 시술용",na=False).values
    F["FA_nontransfer_reason"]=(~is_current).astype("int8")
    mix=NUM(df,"혼합된 난자 수").fillna(0).values; icsi=NUM(df,"미세주입된 난자 수").fillna(0).values
    iemb=NUM(df,"미세주입에서 생성된 배아 수").fillna(0).values; embv=emb.values; both=(mix>0)&(icsi>0)
    with np.errstate(divide="ignore",invalid="ignore"):
        icsi_eff=np.where(icsi>0,iemb/icsi,np.nan); conv_eff=np.where(mix>0,(embv-iemb)/mix,np.nan)
    F["IC_route_eff_diff"]=np.where(both,icsi_eff-conv_eff,np.nan)
    return pd.DataFrame(F,index=df.index)
def expand_lin(D):
    E=pd.DataFrame(index=D.index)
    E["EL_forced"]=(D["EL_set_type"]==1).astype("int8"); E["EL_elective"]=(D["EL_set_type"]==2).astype("int8")
    E["FA_no_transfer"]=D["FA_no_transfer"]; E["FA_nontransfer_reason"]=D["FA_nontransfer_reason"]; E["IC_route_eff_diff"]=D["IC_route_eff_diff"]
    return E
Dtr=build_new_derived(train); Dte=build_new_derived(test); Etr=expand_lin(Dtr); Ete=expand_lin(Dte)
CAND={"tree":{"C1":["EL_set_type"],"C2":["FA_no_transfer","FA_nontransfer_reason"],"C3":["IC_route_eff_diff"]},
      "lin" :{"C1":["EL_forced","EL_elective"],"C2":["FA_no_transfer","FA_nontransfer_reason"],"C3":["IC_route_eff_diff"]}}
ALL_CANDS=["C1","C2","C3"]
def cand_cols(member,cs): return [c for cc in cs for c in CAND[member][cc]]
assert np.array_equal(build_new_derived(train.head(300)).fillna(-9).values, Dtr.head(300).fillna(-9).values), "행단위 독립성 위반!"
print("tf_tree",Xb.shape,"| v2게이팅",V2tr.shape[1],"| lin비율",RATtr.shape[1],"| ✅ 누수안전")

tf_tree (256351, 72) | v2게이팅 11 | lin비율 15 | ✅ 누수안전


## 1. 커밋 base OOF 로드(정렬 확인) + STEP1 (C2 흡수)

In [3]:
def load_commit_oof():
    tr={}
    for p in glob.glob("/kaggle/input/**/oof_predictions.csv",recursive=True):
        d=pd.read_csv(p)
        for k in d.columns:
            kl=k.lower()
            if kl in ("oof_lgb","oof_cat","oof_xgb"): tr[kl.replace("oof_","")]=d[k].values
        break
    return tr
trees0=load_commit_oof()
if len(trees0)>=3:
    for nm in ["lgb","cat","xgb"]: assert roc_auc_score(y,trees0[nm])<0.999, f"{nm} 라벨 오로드 의심"
    op=(runr(trees0["lgb"])+runr(trees0["cat"])+runr(trees0["xgb"]))/3
    txv=NUM(train,"이식된 배아 수").fillna(0).values; m0=(txv==0); pct=rankdata(op)/len(op)
    print(f"커밋 3트리 정렬 확인: base AUC≈{roc_auc_score(y,op):.5f}")
    print(f"STEP1 — 이식=0: n={m0.sum()} ({100*m0.mean():.1f}%) 성공률={y[m0].mean():.4f} | OOF 백분위 중앙값={np.median(pct[m0]):.3f} 하위10%비율={(pct[m0]<0.10).mean():.3f}")
    # 흡수 기준: 이식=0 OOF 백분위 중앙값<0.10 (바닥에 깔림). 기존 '하위10%비율>0.8'은 과도하게 빡빡(거의 흡수도 미흡수로 분류)
    C2_ABSORBED=np.median(pct[m0])<0.10
    print("→ STEP1:", "트리 흡수 → C2 tree-home 자동 제외(진단용, 최종은 그리드 CI)" if C2_ABSORBED else "미흡수 → C2 트리도 후보")
else:
    C2_ABSORBED=False; print("⚠️ 커밋 OOF 없음 — STEP1 생략(C2 흡수 미확정). oof_predictions.csv Add Input 권장.")

커밋 3트리 정렬 확인: base AUC≈0.74010
STEP1 — 이식=0: n=42835 (16.7%) 성공률=0.0196 | OOF 백분위 중앙값=0.084 하위10%비율=0.598
→ STEP1: 트리 흡수 → C2 tree-home 자동 제외(진단용, 최종은 그리드 CI)


## 2. 멤버 학습기 — 트리(v3·v2v3) · 선형(v3·ratio_v3), fold-내부 리크안전

In [4]:
LGP=dict(objective="binary",metric="auc",verbose=-1,learning_rate=0.05,num_leaves=63,feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=1,min_child_samples=50)
XGP=dict(objective="binary:logistic",eval_metric="auc",tree_method="hist",learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8)
def fit_one(kind,Xt,yt,Xv,yv,catf,seed):
    if kind=="lgb": return lgb.train(dict(LGP,seed=seed),lgb.Dataset(Xt,yt,categorical_feature=catf),TREE_ITERS,valid_sets=[lgb.Dataset(Xv,yv,categorical_feature=catf)],callbacks=[lgb.early_stopping(80,verbose=False),lgb.log_evaluation(0)])
    if kind=="xgb": return xgb.train(dict(XGP,seed=seed),xgb.DMatrix(Xt.values,label=yt),TREE_ITERS,evals=[(xgb.DMatrix(Xv.values,label=yv),"v")],early_stopping_rounds=80,verbose_eval=False)
    m=CatBoostClassifier(iterations=TREE_ITERS,learning_rate=0.05,depth=6,verbose=0,random_seed=seed,early_stopping_rounds=80); m.fit(Xt,yt,eval_set=(Xv,yv),cat_features=catf); return m
def pred_one(kind,m,X):
    if kind=="lgb": return m.predict(X)
    if kind=="xgb": return m.predict(xgb.DMatrix(X.values),iteration_range=(0,m.best_iteration+1))
    return m.predict_proba(X)[:,1]
def tree_oof(extra_tr=None,extra_te=None,seed=42,kinds=("lgb","xgb","cat"),full_test=False):
    X=Xb if extra_tr is None else pd.concat([Xb.reset_index(drop=True),extra_tr.reset_index(drop=True)],axis=1)
    Xte=Xb_te if extra_te is None else pd.concat([Xb_te.reset_index(drop=True),extra_te.reset_index(drop=True)],axis=1)
    folds=list(StratifiedKFold(5,shuffle=True,random_state=seed).split(X,y)); catf=[c for c in CATF if c in X.columns]; out={}
    for kind in kinds:
        o=np.zeros(N); tt=np.zeros(len(Xte)) if full_test else None
        for tri,vai in folds:
            m=fit_one(kind,X.iloc[tri],y[tri],X.iloc[vai],y[vai],catf,seed); o[vai]=pred_one(kind,m,X.iloc[vai])
            if full_test: tt+=pred_one(kind,m,Xte)/len(folds)
        out[kind]=(o,tt)
    return out
base_num=Xb.drop(columns=CATF); base_num_te=Xb_te.drop(columns=CATF); TE_COLS=NOMINAL_COLS+[COL_PROC,COL_RSN]
def te_fit(cat,yy,sm=20):
    g=pd.DataFrame({"c":cat.values,"y":yy}).groupby("c")["y"].agg(["mean","count"]); pr=float(yy.mean())
    return ((g["mean"]*g["count"]+pr*sm)/(g["count"]+sm)),pr
def lin_oof(use_ratios=False,extra_tr=None,extra_te=None,seed=42,full_test=False):
    oof=np.zeros(N); tst=np.zeros(len(test)) if full_test else None
    for tri,vai in StratifiedKFold(5,shuffle=True,random_state=seed).split(base_num,y):
        Xt=base_num.iloc[tri].copy(); Xv=base_num.iloc[vai].copy(); Xte=base_num_te.copy() if full_test else None
        for c in TE_COLS:
            enc,pr=te_fit(train[c].astype(str).iloc[tri],y[tri])
            Xt[f"te_{c}"]=train[c].astype(str).iloc[tri].map(enc).fillna(pr).values
            Xv[f"te_{c}"]=train[c].astype(str).iloc[vai].map(enc).fillna(pr).values
            if full_test: Xte[f"te_{c}"]=test[c].astype(str).map(enc).fillna(pr).values
        add_tr=[]; add_te=[]
        if use_ratios: add_tr.append(RATtr); add_te.append(RATte)
        if extra_tr is not None: add_tr.append(extra_tr); add_te.append(extra_te)
        if add_tr:
            Xt=pd.concat([Xt.reset_index(drop=True)]+[a.iloc[tri].reset_index(drop=True) for a in add_tr],axis=1)
            Xv=pd.concat([Xv.reset_index(drop=True)]+[a.iloc[vai].reset_index(drop=True) for a in add_tr],axis=1)
            if full_test: Xte=pd.concat([Xte.reset_index(drop=True)]+[a.reset_index(drop=True) for a in add_te],axis=1)
        med=Xt.median(); Xt=Xt.fillna(med); Xv=Xv.fillna(med)
        sc=StandardScaler().fit(Xt); m=LogisticRegression(max_iter=2000,C=0.5).fit(sc.transform(Xt),y[tri])
        oof[vai]=m.predict_proba(sc.transform(Xv))[:,1]
        if full_test: tst+=m.predict_proba(sc.transform(Xte.fillna(med)))[:,1]/5
    return (oof,tst) if full_test else oof
print("학습기 준비")

학습기 준비


## 3. 스크리닝 (필터) — 트리(lgb)·선형 멀티시드 Δ → 통과 (후보×멤버)

In [5]:
def gate(mu,sd): return (abs(mu)>sd and mu>0.0002) if len(SEEDS)>1 else (mu>0.0002)
TREE_KINDS=("lgb","cat","xgb")
# 모델별 base OOF (시드별) 캐시
tree_b={k:{s:None for s in SEEDS} for k in TREE_KINDS}
for s in SEEDS:
    r=tree_oof(seed=s,kinds=TREE_KINDS)
    for k in TREE_KINDS: tree_b[k][s]=r[k][0]
lin_b={s:lin_oof(seed=s) for s in SEEDS}
screen={}; pass_model={k:[] for k in TREE_KINDS}; pass_lin=[]
print("=== 스크리닝 (필터) — 트리 3종 모델별 + 선형 ===")
for cand in ALL_CANDS:
    screen[cand]={}
    # 트리 3종 각각 단독 멀티시드 Δ (한 번 +feat OOF 생성해 3종 동시 평가)
    feat_oof={k:{} for k in TREE_KINDS}
    for s in SEEDS:
        rf=tree_oof(extra_tr=Dtr[cand_cols("tree",[cand])],seed=s,kinds=TREE_KINDS)
        for k in TREE_KINDS: feat_oof[k][s]=rf[k][0]
    line=[]
    for k in TREE_KINDS:
        dl=[roc_auc_score(y,feat_oof[k][s])-roc_auc_score(y,tree_b[k][s]) for s in SEEDS]
        mu,sd=float(np.mean(dl)),float(np.std(dl)); screen[cand][k]=(mu,sd)
        pk=gate(mu,sd)   # STEP1은 보고만, 모델별 Δ가 결정(집계 신호로 모델별 판단 덮지 않음 — lgb 프록시 폐기와 동일 원칙)
        if pk: pass_model[k].append(cand)
        line.append(f"{k} Δ={mu:+.5f}±{sd:.5f} {'✅' if pk else '—'}")
    # 선형
    dn=[roc_auc_score(y,lin_oof(extra_tr=Etr[cand_cols('lin',[cand])],seed=s))-roc_auc_score(y,lin_b[s]) for s in SEEDS]
    lmu,lsd=float(np.mean(dn)),float(np.std(dn)); screen[cand]["lin"]=(lmu,lsd); pl=gate(lmu,lsd)
    if pl: pass_lin.append(cand)
    line.append(f"lin Δ={lmu:+.5f}±{lsd:.5f} {'✅' if pl else '—'}")
    print(f"{cand}: "+" | ".join(line)+(("  (STEP1:블렌드 흡수 — 모델별 Δ로 판단)" if cand=='C2' and C2_ABSORBED else "")))
print("\n통과 — lgb:",pass_model['lgb'] or "없음","| cat:",pass_model['cat'] or "없음","| xgb:",pass_model['xgb'] or "없음","| lin:",pass_lin or "없음")

=== 스크리닝 (필터) — 트리 3종 모델별 + 선형 ===
C1: lgb Δ=-0.00006±0.00009 — | cat Δ=+0.00003±0.00006 — | xgb Δ=-0.00018±0.00011 — | lin Δ=+0.00076±0.00000 ✅
C2: lgb Δ=+0.00000±0.00009 — | cat Δ=+0.00005±0.00005 — | xgb Δ=-0.00005±0.00010 — | lin Δ=+0.01257±0.00001 ✅  (STEP1:블렌드 흡수 — 모델별 Δ로 판단)
C3: lgb Δ=-0.00011±0.00008 — | cat Δ=-0.00003±0.00006 — | xgb Δ=-0.00019±0.00014 — | lin Δ=+0.00001±0.00001 —

통과 — lgb: 없음 | cat: 없음 | xgb: 없음 | lin: ['C1', 'C2']


## 4. 통과 멤버의 v3·v2+v3 (선형: v3·ratio+v3) 뷰 OOF/test 생성 → 그리드 합류용 저장
멤버별 통과 후보를 합쳐(union) 한 뷰. seed42(그리드 기준). SEEDS 멀티시드면 _s{seed} 추가 저장.

In [6]:
sfx="_DRYRUN" if DRYRUN else ""
saved=[]
# ── 트리: 모델별 통과 후보 union으로 그 모델의 v3 / v2v3 OOF/test 생성 (통과 모델만)
def gen_tree_view(kind, cols, with_v2):
    etr=Dtr[cols] if not with_v2 else pd.concat([V2tr.reset_index(drop=True),Dtr[cols].reset_index(drop=True)],axis=1)
    ete=Dte[cols] if not with_v2 else pd.concat([V2te.reset_index(drop=True),Dte[cols].reset_index(drop=True)],axis=1)
    r=tree_oof(extra_tr=etr,extra_te=ete,seed=42,kinds=(kind,),full_test=True)
    return r[kind][0], r[kind][1]
TREE_KINDS=("lgb","cat","xgb")
v3_oof={}; v3_tst={}; v2v3_oof={}; v2v3_tst={}
for kind in TREE_KINDS:
    if not pass_model[kind]: 
        print(f"  트리 {kind}: 통과 없음 → 생성 skip"); continue
    cols=cand_cols("tree",pass_model[kind]); t0=time.time()
    o3,t3=gen_tree_view(kind,cols,False); v3_oof[kind]=o3; v3_tst[kind]=t3
    o23,t23=gen_tree_view(kind,cols,True); v2v3_oof[kind]=o23; v2v3_tst[kind]=t23
    print(f"  트리 {kind} [{pass_model[kind]}]: v3={roc_auc_score(y,o3):.5f} | v2v3={roc_auc_score(y,o23):.5f} ({(time.time()-t0)/60:.1f}분)")
if v3_oof:
    dfo={"y":y}; dft={"ID":test[ID_COL].values}
    for k in v3_oof: dfo[f"oof_v3_{k}"]=v3_oof[k]; dft[f"v3_{k}"]=v3_tst[k]
    pd.DataFrame(dfo).to_csv(f"/kaggle/working/oof_v3_trees{sfx}.csv",index=False)
    pd.DataFrame(dft).to_csv(f"/kaggle/working/test_v3_trees{sfx}.csv",index=False); saved+=["oof_v3_trees","test_v3_trees"]
if v2v3_oof:
    dfo={"y":y}; dft={"ID":test[ID_COL].values}
    for k in v2v3_oof: dfo[f"oof_v2v3_{k}"]=v2v3_oof[k]; dft[f"v2v3_{k}"]=v2v3_tst[k]
    pd.DataFrame(dfo).to_csv(f"/kaggle/working/oof_v2v3_trees{sfx}.csv",index=False)
    pd.DataFrame(dft).to_csv(f"/kaggle/working/test_v2v3_trees{sfx}.csv",index=False); saved+=["oof_v2v3_trees","test_v2v3_trees"]
if not v3_oof: print("  트리 통과 모델 없음 → 트리 뷰 미생성")
# ── 선형: v3 = base+v3 / ratio_v3 = base+ratio+v3 (통과 후보 union)
if pass_lin:
    lcols=cand_cols("lin",pass_lin); le_tr=Etr[lcols]; le_te=Ete[lcols]
    o,t=lin_oof(use_ratios=False,extra_tr=le_tr,extra_te=le_te,seed=42,full_test=True)
    pd.DataFrame({"oof_lin_v3":o,"y":y}).to_csv(f"/kaggle/working/oof_lin_v3{sfx}.csv",index=False)
    pd.DataFrame({"ID":test[ID_COL],"lin_v3":t}).to_csv(f"/kaggle/working/test_lin_v3{sfx}.csv",index=False); saved+=["oof_lin_v3","test_lin_v3"]
    o2,t2=lin_oof(use_ratios=True,extra_tr=le_tr,extra_te=le_te,seed=42,full_test=True)
    pd.DataFrame({"oof_lin_ratio_v3":o2,"y":y}).to_csv(f"/kaggle/working/oof_lin_ratio_v3{sfx}.csv",index=False)
    pd.DataFrame({"ID":test[ID_COL],"lin_ratio_v3":t2}).to_csv(f"/kaggle/working/test_lin_ratio_v3{sfx}.csv",index=False); saved+=["oof_lin_ratio_v3","test_lin_ratio_v3"]
    print(f"  선형 [{pass_lin}]: v3={roc_auc_score(y,o):.5f} | ratio_v3={roc_auc_score(y,o2):.5f}")
else: print("  선형 통과 없음 → 선형 v3 생성 skip")
json.dump({"screen":screen,"pass_model":pass_model,"pass_lin":pass_lin,"C2_absorbed":bool(C2_ABSORBED),
           "tree_v3_cols":{k:cand_cols("tree",pass_model[k]) for k in TREE_KINDS},"lin_v3_cols":cand_cols("lin",pass_lin),"dryrun":DRYRUN},
          open(f"/kaggle/working/v3b_screen{sfx}.json","w"),ensure_ascii=False,default=float)
print("\n💾 저장:",saved,"+ v3b_screen.json")
print("→ 그리드에 Add Input → 모델별 {base,v2,v3,v2v3} 선택. 부분 존재 뷰(예: cat_v3만)도 그리드가 처리.")

  트리 lgb: 통과 없음 → 생성 skip
  트리 cat: 통과 없음 → 생성 skip
  트리 xgb: 통과 없음 → 생성 skip
  트리 통과 모델 없음 → 트리 뷰 미생성
  선형 [['C1', 'C2']]: v3=0.72771 | ratio_v3=0.73084

💾 저장: ['oof_lin_v3', 'test_lin_v3', 'oof_lin_ratio_v3', 'test_lin_ratio_v3'] + v3b_screen.json
→ 그리드에 Add Input → 모델별 {base,v2,v3,v2v3} 선택. 부분 존재 뷰(예: cat_v3만)도 그리드가 처리.


## 5. 스크리닝 표

In [7]:
rows=[]
for cand in ALL_CANDS:
    sc=screen[cand]
    row={"후보":cand}
    for k in ("lgb","cat","xgb"): row[f"{k} Δ"]=f"{sc[k][0]:+.5f}"
    row["lin Δ"]=f"{sc['lin'][0]:+.5f}"
    row["통과(트리)"]=",".join([k for k in ("lgb","cat","xgb") if cand in pass_model[k]]) or "—"
    row["lin통과"]="✅" if cand in pass_lin else "—"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("\nC2 STEP1:",("트리 흡수" if C2_ABSORBED else "트리 미흡수"),"| DRYRUN:",DRYRUN)
print("주의: 스크리닝=필터. 채택 결정은 그리드 'v3 허용 vs 불허' CI. 트리 3종 각각 봄(lgb 프록시 폐기 — v2가 모델별로 달랐던 증거 반영).")

후보    lgb Δ    cat Δ    xgb Δ    lin Δ 통과(트리) lin통과
C1 -0.00006 +0.00003 -0.00018 +0.00076      —     ✅
C2 +0.00000 +0.00005 -0.00005 +0.01257      —     ✅
C3 -0.00011 -0.00003 -0.00019 +0.00001      —     —

C2 STEP1: 트리 흡수 | DRYRUN: False
주의: 스크리닝=필터. 채택 결정은 그리드 'v3 허용 vs 불허' CI. 트리 3종 각각 봄(lgb 프록시 폐기 — v2가 모델별로 달랐던 증거 반영).


## 6. [추가] 트리 v3 / v2v3 강제 생성 — 스크리닝 통과 무관, 그리드가 4뷰 다 보게
단독 Δ가 노이즈여도 블렌드에선 다를 수 있음(과거 '미미해도 제출→최고' 철학). winner's curse는 그리드 'v3 허용 vs 불허' CI + LB로 방어.

In [8]:
# ── [추가] 트리 v3 / v2v3 강제 생성 (스크리닝 통과 무관) — 그리드가 4뷰 다 보게
# 철학: 단독 Δ가 노이즈여도 블렌드에선 다를 수 있음 → 만들어서 그리드+CI가 판정 (lgb 프록시 폐기와 동일 원칙)
# C3 폐기 후 후보 = C1·C2. 트리 union = C1+C2 한 세트. CatBoost 포함이라 ~1시간.
sfx="_DRYRUN" if DRYRUN else ""
FORCE_TREE_CANDS=["C1","C2"]                      # 강제 생성 후보 (스크리닝 무관). ALL_CANDS와 동일.
tcols=cand_cols("tree",FORCE_TREE_CANDS)
print("트리 강제 생성 후보:",FORCE_TREE_CANDS,"| 컬럼:",tcols)
def _gen_tree_view(kind, cols, with_v2):
    etr=Dtr[cols] if not with_v2 else pd.concat([V2tr.reset_index(drop=True),Dtr[cols].reset_index(drop=True)],axis=1)
    ete=Dte[cols] if not with_v2 else pd.concat([V2te.reset_index(drop=True),Dte[cols].reset_index(drop=True)],axis=1)
    r=tree_oof(extra_tr=etr,extra_te=ete,seed=42,kinds=(kind,),full_test=True)
    return r[kind][0], r[kind][1]
TREE_KINDS=("lgb","cat","xgb")
v3o={}; v3t={}; v23o={}; v23t={}
import time as _t
for kind in TREE_KINDS:
    t0=_t.time()
    o3,t3=_gen_tree_view(kind,tcols,False); v3o[kind]=o3; v3t[kind]=t3
    o23,t23=_gen_tree_view(kind,tcols,True); v23o[kind]=o23; v23t[kind]=t23
    print(f"  트리 {kind}: v3={roc_auc_score(y,o3):.5f} | v2v3={roc_auc_score(y,o23):.5f} ({(_t.time()-t0)/60:.1f}분)")
# 저장 (그리드 MAP과 동일 컬럼명)
pd.DataFrame({"oof_v3_lgb":v3o["lgb"],"oof_v3_cat":v3o["cat"],"oof_v3_xgb":v3o["xgb"],"y":y}).to_csv(f"/kaggle/working/oof_v3_trees{sfx}.csv",index=False)
pd.DataFrame({"ID":test[ID_COL],"v3_lgb":v3t["lgb"],"v3_cat":v3t["cat"],"v3_xgb":v3t["xgb"]}).to_csv(f"/kaggle/working/test_v3_trees{sfx}.csv",index=False)
pd.DataFrame({"oof_v2v3_lgb":v23o["lgb"],"oof_v2v3_cat":v23o["cat"],"oof_v2v3_xgb":v23o["xgb"],"y":y}).to_csv(f"/kaggle/working/oof_v2v3_trees{sfx}.csv",index=False)
pd.DataFrame({"ID":test[ID_COL],"v2v3_lgb":v23t["lgb"],"v2v3_cat":v23t["cat"],"v2v3_xgb":v23t["xgb"]}).to_csv(f"/kaggle/working/test_v2v3_trees{sfx}.csv",index=False)
print("\n💾 저장 완료: oof_v3_trees / test_v3_trees / oof_v2v3_trees / test_v2v3_trees", f"(접미사 '{sfx}')" if sfx else "")
print("→ 선형(이미 생성된 oof_lin_v3/oof_lin_ratio_v3)과 함께 그리드 Add Input")


트리 강제 생성 후보: ['C1', 'C2'] | 컬럼: ['EL_set_type', 'FA_no_transfer', 'FA_nontransfer_reason']
  트리 lgb: v3=0.73951 | v2v3=0.73965 (1.0분)
  트리 cat: v3=0.73960 | v2v3=0.73974 (19.4분)
  트리 xgb: v3=0.73972 | v2v3=0.73964 (1.7분)

💾 저장 완료: oof_v3_trees / test_v3_trees / oof_v2v3_trees / test_v2v3_trees 
→ 선형(이미 생성된 oof_lin_v3/oof_lin_ratio_v3)과 함께 그리드 Add Input
